# HealthyAir - Feature Engineering Notebook
Run these notebooks in order: exploration → cleaning → feature engineering, keeping the raw files unchanged.

## Feature engineering
The notebook creates time and lag features, measurements available at hour *t* to predict the NO2 measurement/category at hour *t+1*. 

In [ ]:
# Import Libraries
import sys
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import statsmodels
import scipy
import os
import gcsfs
import warnings
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    f1_score
)

# Confirm location
print("Location:", sys.executable)

SEED = 42
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid", context="notebook")

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seaborn:", sns.__version__)
print("statsmodels:", statsmodels.__version__)

Location: /home/chloekc/datahandling/healthyair/.venv/bin/python3
Python: 3.12.3
NumPy: 2.5.3
pandas: 3.0.6
SciPy: 1.18.1
Matplotlib: 3.11.1
Seaborn: 0.13.2
statsmodels: 0.15.0


## Load Data & Validate

In [2]:
# Load data from Google Cloud Storage
GCS_PROJECT_ID = "healthy_air_analysis"
GCS_BUCKET = "uci_air_quality_20119102"
GCS_FILE = "gs://uci_air_quality_20119102/data_raw/AirQualityUCI.csv"
df = pd.read_csv(GCS_FILE)

# Data contract
required = {"Date", "Time", "CO(GT)", "PT08.S1(CO)", "NMHC(GT)", "C6H6(GT)",
            "PT08.S2(NMHC)", "NOx(GT)", "PT08.S3(NOx)", "NO2(GT)", "PT08.S4(NO2)", "PT08.S5(O3)", "T", "RH", "AH"}
assert required.issubset(df.columns)
#assert set(df["Quality"].unique()).issubset({0, 1, 3})
print("Initial data-contract checks passed.")

print("Dataset path:", GCS_BUCKET)
print(f"Rows: {df.shape[0]:,}; columns: {df.shape[1]}")
display(df.head())

df_target = df.copy()

Initial data-contract checks passed.
Dataset path: uci_air_quality_20119102
Rows: 9,471; columns: 17


,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Unnamed: 15,Unnamed: 16
0,3/10/2004,18:00:00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578,NaN,NaN
1,3/10/2004,19:00:00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255,NaN,NaN
2,3/10/2004,20:00:00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502,NaN,NaN
3,3/10/2004,21:00:00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867,NaN,NaN
4,3/10/2004,22:00:00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888,NaN,NaN


## 1. Define the next-hour target
A simple `.shift(-1)` is only a true next-hour target if the next row is exactly one hour later. Mask gaps to avoid accidentally predicting several hours ahead.

In [ ]:
# NO2 thresholds (µg/m³)
GOOD_THRESHOLD = 100
POOR_THRESHOLD = 200

# Ensure NO2 is numeric
df_target["NO2(GT)"] = pd.to_numeric(
    df["NO2(GT)"], errors="coerce"
)

# Convert UCI missing-value marker
df_target["NO2(GT)"] = df_target["NO2(GT)"].replace(-200, np.nan)

# Create three air-quality categories
conditions = [
    df_target["NO2(GT)"] < GOOD_THRESHOLD,
    (df_target["NO2(GT)"] >= GOOD_THRESHOLD) &
    (df_target["NO2(GT)"] <= POOR_THRESHOLD),
    df_target["NO2(GT)"] > POOR_THRESHOLD
]

labels = ["GOOD", "MODERATE", "POOR"]

df_target["NO2_Category"] = np.select(
    conditions,
    labels,
    default=None
)

# Check results
print(df_target[["NO2(GT)", "NO2_Category"]].head(10))

print("\nCategory counts:")
print(df_target["NO2_Category"].value_counts(dropna=False))

print("\nCategory percentages:")
print(
    df_target["NO2_Category"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)


In [ ]:
plt.figure(figsize=(9, 5))

sns.countplot(
    data=df_target,
    x="NO2_Category",
    order=["GOOD", "MODERATE", "POOR"]
)

plt.title("Distribution of NO2 Air Quality Categories")
plt.xlabel("NO2 Category")
plt.ylabel("Number of observations")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
df_target["Target_Next_Hour"] = df_target["NO2_Category"].shift(-1)

In [ ]:
# Select numerical measurement variables
measurement_cols = [
    "CO(GT)", "PT08.S1(CO)", "NMHC(GT)", "C6H6(GT)",
    "PT08.S2(NMHC)", "NOx(GT)", "PT08.S3(NOx)",
    "NO2(GT)", "PT08.S4(NO2)", "PT08.S5(O3)",
    "T", "RH", "AH"
]

In [ ]:
# Define project-specific NO2 categories
forecast_df["Target_Next_Hour"] = pd.cut(
    forecast_df["NO2_next_hour"],
    bins=[-np.inf, 100, 200, np.inf],
    labels=["GOOD", "MODERATE", "POOR"]
)

# Candidate predictors: current measurements only
predictors = measurement_cols.copy()

# Exclude NMHC initially if its missingness is excessive.
# Review the missingness results before deciding.
# predictors.remove("NMHC(GT)")

# Prepare modelling dataset
model_df = forecast_df[
    ["Datetime"] + predictors + ["Target_Next_Hour"]
].dropna(subset=["Target_Next_Hour"])

# Chronological 80/20 train-test split
split_index = int(len(model_df) * 0.8)

train = model_df.iloc[:split_index]
test = model_df.iloc[split_index:]

X_train = train[predictors]
X_test = test[predictors]

y_train = train["Target_Next_Hour"]
y_test = test["Target_Next_Hour"]

# Remove predictors completely missing in the training data
usable_predictors = X_train.columns[
    X_train.notna().any()
].tolist()

X_train = X_train[usable_predictors]
X_test = X_test[usable_predictors]

# Baseline: always predict the most frequent training category
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

# Random Forest pipeline
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)

rf_predictions = rf_pipeline.predict(X_test)

# Compare model performance
results = pd.DataFrame({
    "model": ["Baseline", "Random Forest"],
    "accuracy": [
        baseline.score(X_test, y_test),
        rf_pipeline.score(X_test, y_test)
    ],
    "balanced_accuracy": [
        balanced_accuracy_score(y_test, baseline_predictions),
        balanced_accuracy_score(y_test, rf_predictions)
    ],
    "macro_f1": [
        f1_score(y_test, baseline_predictions, average="macro"),
        f1_score(y_test, rf_predictions, average="macro")
    ]
})

display(results.round(3))

# Detailed Random Forest results
print("Random Forest classification report:")
print(classification_report(
    y_test,
    rf_predictions,
    zero_division=0
))
